# 🚀 A Corrida do Processamento
Bem-vindos à dinâmica de Sistemas Distribuídos e Paralelos!

Neste laboratório, vamos testar na prática a diferença entre **Concorrência**, **Paralelismo** e **Sistemas Distribuídos**.
Para não dependermos de arquivos externos, nossa "tarefa pesada" será puramente matemática e exigirá muito da CPU: **contar números primos em um grande intervalo**.


In [ ]:
import time
import threading
import multiprocessing
import requests
import platform
import math

# Função que simula uma carga de processamento pesado (CPU-bound)
def trabalho_pesado(id_tarefa, inicio, fim):
    print(f"[{id_tarefa}] Iniciando busca de primos de {inicio} até {fim}...")
    primos = 0
    # Algoritmo intencionalmente custoso para estressar a CPU
    for num in range(inicio, fim):
        if num > 1:
            is_prime = True
            for i in range(2, int(math.sqrt(num)) + 1):
                if num % i == 0:
                    is_prime = False
                    break
            if is_prime:
                primos += 1
    print(f"[{id_tarefa}] Concluído! Primos encontrados: {primos}")
    return primos

INTERVALO = 3_000_000 # Tamanho total do problema. Aumente se o PC for muito rápido!

## ⚖️ Fase 0: Código sequencial

Vamos calcular os números primos usando programação convencional, na qual é utilizada uma única thread.

In [ ]:
inicio_tempo = time.time()

trabalho_pesado("Thread-única", 0, INTERVALO)

fim_tempo = time.time()
print(f"\n⏱️ TEMPO TOTAL CONCORRENTE: {fim_tempo - inicio_tempo:.2f} segundos")

## ⏳ Fase 1: Concorrência (Threads)
A concorrência lida com várias coisas "ao mesmo tempo" alternando entre elas (troca de contexto).
No Python, devido a uma trava chamada **GIL (Global Interpreter Lock)**, as *Threads* não conseguem usar múltiplos núcleos do processador para cálculos matemáticos. Elas disputam o mesmo núcleo.
Execute a célula abaixo e observe o tempo que leva.

In [ ]:
inicio_tempo = time.time()

# Dividindo o problema em tarefas concorrentes
threads = []
num_threads = 2
passo = INTERVALO // num_threads

for i in range(num_threads):
    t = threading.Thread(target=trabalho_pesado, args=(f"Thread-{i+1}", i*passo, (i+1)*passo))
    threads.append(t)
    t.start()

for t in threads:
    t.join() # Espera todas as threads terminarem

fim_tempo = time.time()
print(f"\n⏱️ TEMPO TOTAL CONCORRENTE: {fim_tempo - inicio_tempo:.2f} segundos")

## ⚡ Fase 2: Paralelismo (Múltiplos Núcleos)
Agora vamos aplicar o paralelismo real usando a biblioteca `multiprocessing`.
Isso cria processos independentes no sistema operacional, permitindo que cada pedaço do código rode em um **núcleo físico diferente** do seu processador, ao mesmo exato tempo.
Execute e compare a diferença de tempo!

In [ ]:
# A verificação __main__ é necessária no Windows para multiprocessing
if __name__ == '__main__':
    inicio_tempo = time.time()

    processos = []
    num_processos = multiprocessing.cpu_count()
    passo = INTERVALO // num_processos

    for i in range(2):
        p = multiprocessing.Process(target=trabalho_pesado, args=(f"Processo-{i+1}", i*passo, (i+1)*passo))
        processos.append(p)
        p.start()

    for p in processos:
        p.join() # Espera todos os processos terminarem

    fim_tempo = time.time()
    print(f"\n🚀 TEMPO TOTAL PARALELO: {fim_tempo - inicio_tempo:.2f} segundos")

## 🌍 Fase 3: Computação Distribuída
O problema cresceu. Calcular o intervalo inteiro travaria sua máquina se feito sozinho.
A partir de agora, o seu computador inteiro é apenas **um único Nó** em um cluster gigante (a sala de aula).
Sua missão é processar apenas uma fatia pequena dos dados e enviar o resultado através da rede para o "Nó Mestre" (o painel do professor).

In [ ]:
import random

# ⚠️ Acesse https://webhook.site/ e cole a sua "Your unique URL" aqui abaixo:
URL_DO_SERVIDOR = "URL_WEBHOOK"

ALUNO = "Nome_do_Aluno" # ⬅️ Troque para o seu nome real ou apelido!

print(f"Nó [{ALUNO}] processando lote localmente...")

# Definimos o limite mínimo e máximo geral que queremos
limite_inferior = 1
limite_superior = INTERVALO

# Sorteamos o número de início
inicio_aleatorio = random.randint(limite_inferior, limite_superior - 1000)

# Sorteamos o número de fim, garantindo que seja MAIOR que o início
# Adicionamos um salto mínimo (ex: +1000) para garantir que haja um processamento real
fim_aleatorio = random.randint(inicio_aleatorio + 1000, limite_superior)

print(f"O intervalo sorteado foi: de {inicio_aleatorio} até {fim_aleatorio}")

# Cada aluno processa um lote pequeno e rápido
resultado_local = trabalho_pesado(f"Nó-{ALUNO}", inicio_aleatorio, fim_aleatorio)

# Monta o pacote de dados para trafegar na rede (JSON)
pacote_de_dados = {
    "aluno": ALUNO,
    "maquina": platform.node(),
    "sistema": platform.system(),
    "resultado": resultado_local,
    "status": "Pronto para próxima tarefa!"
}

print("\n📡 Enviando dados pela rede para o Nó Mestre...")

try:
    # A comunicação via rede é o coração dos sistemas distribuídos!
    resposta = requests.post(URL_DO_SERVIDOR, json=pacote_de_dados, timeout=5)

    if resposta.status_code == 200:
        print("✅ Sucesso! O servidor central recebeu seus dados.")
    else:
        print(f"⚠️ O servidor respondeu, mas com erro de código: {resposta.status_code}")

except Exception as erro:
    # Aqui simulamos o que acontece quando a rede cai ou um nó perde a conexão
    print("❌ Falha na rede. O nó central caiu ou você está sem internet? Erro:", erro)